In [1]:
import re
import os, time, pickle
import torch
from omegaconf import OmegaConf
import hydra
import logging
from rfdiffusion.util import writepdb_multi, writepdb
from rfdiffusion.inference import utils as iu
from rfdiffusion.chemical import seq2chars
from rfdiffusion.kinematics import get_init_xyz, xyz_to_t2d
from hydra.core.hydra_config import HydraConfig
import numpy as np
import torch.nn.functional as nn
import random
import glob
from adabelief_pytorch import AdaBelief
import sys

from pathlib import Path

/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/rfdiffusion/util.py:308: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at ../aten/src/ATen/native/Cross.cpp:62.)
  Z = torch.cross(Xn, Yn)
/home/ics121/miniforge3/envs/SE3nv-test/lib/python3.10/site-packages/dgl/backend/pytorch/sparse.py:104: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=th.float16)
/home/ics121/miniforge3/envs/SE3nv-test/lib/python3.10/site-packages/dgl/backend/pytorch/sparse.py:128: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, dZ):
/home/ics121/miniforge3/envs/SE3nv-te

In [34]:
# Copyright AstraZeneca UK Ltd. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Inversion script, adapted for use as a function inside a notebook.

Original: run_inversion.py (standalone, driven by @hydra.main)
Adapted from run_inference.py (RFdiffusion) by Kevin Michalewicz (Imperial/UBA)

Usage from a notebook cell:

    from run_inversion_notebook import run_inversion

    run_inversion(
        config_path="../config/inference",   # folder containing base.yaml
        config_name="base",
        overrides=["inference.num_designs=5"],  # optional hydra-style overrides
        output_dir="./outputs/my_run",          # where pdbs / embeddings are written
    )

No logic from the original reverse-diffusion / optimisation loop has been
changed. What changed:
  * the @hydra.main entrypoint was removed and replaced with an explicit
    hydra.compose() call inside the function, so config loading no longer
    needs a module-level decorator or a CLI invocation;
  * hydra.main() used to silently chdir into a per-run output directory
    which is where all the relative paths (output_prefix, learnt_embeddings/...)
    used to land. Since we no longer get that implicit chdir in a notebook,
    an explicit `output_dir` argument is introduced and all previously
    relative file I/O (pdb outputs, learnt_embeddings/*.pt) is resolved
    against it instead.
"""

import re
import os, time, pickle
import torch
from omegaconf import OmegaConf, DictConfig
import hydra
from hydra import initialize_config_dir, compose
import logging
from rfdiffusion.util import writepdb_multi, writepdb
from rfdiffusion.inference import utils as iu
from rfdiffusion.chemical import seq2chars
from rfdiffusion.kinematics import get_init_xyz, xyz_to_t2d
import numpy as np
import torch.nn.functional as nn
import random
import glob
from adabelief_pytorch import AdaBelief
import sys

log = logging.getLogger(__name__)
if not log.handlers:
    logging.basicConfig(level=logging.INFO)


def make_deterministic(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def run_inversion(
    conf: DictConfig = None,
    config_path: str = "../config/inference",
    config_name: str = "base",
    overrides: list = None,
    output_dir: str = None,
) -> None:
    """
    Run the inversion / reverse-diffusion loop.

    Parameters
    ----------
    conf : DictConfig, optional
        An already-built config (e.g. from a previous hydra.compose() call,
        or OmegaConf.load()). If given, config_path/config_name/overrides
        are ignored.
    config_path : str
        Path (absolute, or relative to this file's directory) to the folder
        holding the hydra config (mirrors the original
        `config_path="../config/inference"` from the @hydra.main decorator).
    config_name : str
        Name of the yaml config file (without extension) inside config_path.
    overrides : list[str], optional
        Hydra-style dotlist overrides, e.g. ["inference.num_designs=5"].
    output_dir : str, optional
        Directory that all outputs (pdb files, learnt_embeddings/*.pt) are
        written into. Replaces the implicit chdir that @hydra.main used to
        perform. Defaults to the current working directory if not given.
    """

    # ------------------------------------------------------------------
    # Config loading (replaces the @hydra.main(config_path=..., config_name=...)
    # decorator which previously built `conf` before calling main()).
    # ------------------------------------------------------------------
    if conf is None:
        cfg_dir = config_path
        if not os.path.isabs(cfg_dir):
            cfg_dir = os.path.abspath(
                os.path.join(os.path.dirname(os.path.abspath(__file__)), config_path)
            )
        with initialize_config_dir(version_base=None, config_dir=cfg_dir):
            conf = compose(config_name=config_name, overrides=overrides or [])

    # ------------------------------------------------------------------
    # Output location (replaces hydra's implicit per-run chdir).
    # ------------------------------------------------------------------
    if output_dir is None:
        output_dir = os.getcwd()
    output_dir = os.path.abspath(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    embeddings_dir = os.path.join(output_dir, "learnt_embeddings")
    os.makedirs(embeddings_dir, exist_ok=True)

    conf.inference.deterministic = False
    if conf.inference.deterministic:
        print('Deterministic?')
        #make_deterministic()

    # Check for available GPU and print result of check
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(torch.cuda.current_device())
        log.info(f"Found GPU with device_name {device_name}. Will run RFdiffusion on {device_name}")
    else:
        log.info("////////////////////////////////////////////////")
        log.info("///// NO GPU DETECTED! Falling back to CPU /////")
        log.info("////////////////////////////////////////////////")

    # Initialize sampler and target/contig.
    sampler = iu.sampler_selector(conf)

    # Loop over number of designs to sample.
    output_prefix_full = os.path.join(output_dir, sampler.inf_conf.output_prefix)
    os.makedirs(os.path.dirname(output_prefix_full), exist_ok=True)

    design_startnum = sampler.inf_conf.design_startnum
    if sampler.inf_conf.design_startnum == -1:
        existing = glob.glob(output_prefix_full + "*.pdb")
        indices = [-1]
        for e in existing:
            print(e)
            m = re.match(".*_(\d+)\.pdb$", e)
            print(m)
            if not m:
                continue
            m = m.groups()[0]
            indices.append(int(m))
        design_startnum = max(indices) + 1
    inverse_ms = True # False
    loss_fn = torch.nn.MSELoss()
    loss_fn_2 = torch.nn.MSELoss()
    x_init, seq_init = sampler.sample_init(inverse_ms=inverse_ms)
    original_roi_indices = sampler.mappings['con_ref_idx0']
    # original_roi_indices = sampler.mappings['complex_con_ref_idx0']
    len_roi_indices = len(original_roi_indices)

    # Initialise the msa, pair, and state embeddings
    if sampler.inf_conf.inversion_train:
        msa_prev = torch.zeros((1, len_roi_indices, 256), device=sampler.device, requires_grad=True)
        pair_prev = torch.zeros((1, len_roi_indices, len_roi_indices, 128), device=sampler.device, requires_grad=True)
        state_prev = torch.zeros((1, len_roi_indices, 64), device=sampler.device, requires_grad=True)
    else:
        msa_prev = torch.load(os.path.join(embeddings_dir, f'msa_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
        state_prev = torch.load(os.path.join(embeddings_dir, f'state_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
        pair_prev = torch.load(os.path.join(embeddings_dir, f'pair_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
    loss_list = []

    for i_des in range(design_startnum, design_startnum + sampler.inf_conf.num_designs):
        #if conf.inference.deterministic:
            #make_deterministic(i_des)

        if not sampler.inf_conf.inversion_train:
            g = torch.Generator(device='cpu').manual_seed(i_des)
            random_rate = (torch.randint(0, 21, (1,), generator=g) * 0.05).item()
            row_masking = bool(torch.randint(0, 2, (), generator=g))
        else:
            random_rate = 0
            row_masking = True

        start_time = time.time()
        #out_prefix = f"{sampler.inf_conf.input_pdb[11:-4]}_{sampler.inf_conf.output_prefix}_{i_des}"
        out_prefix = os.path.join(
            output_dir, f"{sampler.inf_conf.input_pdb[11:-4]}_{sampler.inf_conf.output_prefix}_{i_des}"
        )
        log.info(f"Making design {out_prefix} with random rate {random_rate} and {'row' if row_masking else 'column'} masking")
        if sampler.inf_conf.cautious and os.path.exists(out_prefix + ".pdb"):
            log.info(
                f"(cautious mode) Skipping this design because {out_prefix}.pdb already exists."
            )
            continue
        x_init, seq_init = sampler.sample_init(inverse_ms=inverse_ms)
        if not sampler.inf_conf.inversion_train:
            msa_prev = torch.load(os.path.join(embeddings_dir, f'msa_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
            state_prev = torch.load(os.path.join(embeddings_dir, f'state_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
            pair_prev = torch.load(os.path.join(embeddings_dir, f'pair_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
        if i_des != design_startnum and sampler.inf_conf.inversion_train:
            x_init, seq_init = sampler.sample_init(inverse_ms=inverse_ms)
            prev_roi_indices = torch.clone(roi_indices)
        else:
            prev_roi_indices = torch.arange(len_roi_indices)

        roi_indices = sampler.mask_seq.nonzero()[:, 1]
        #if conf.inference.mutation_pos:
        #    seq_init[roi_indices[conf.inference.mutation_pos[0]], -1] = 1
        #    seq_init[roi_indices[conf.inference.mutation_pos[0]], :-1] = 0

        denoised_xyz_stack = []
        px0_xyz_stack = []
        seq_stack = []
        plddt_stack = []

        x_t = torch.clone(x_init).detach()
        seq_t = torch.clone(seq_init).detach()

        msa_prev = msa_prev.detach()
        pair_prev = pair_prev.detach()
        state_prev = state_prev.detach()

        msa_prev_ = torch.zeros((msa_prev.shape[0], seq_t.shape[0], msa_prev.shape[-1]), device=sampler.device)
        pair_prev_ = torch.zeros((pair_prev.shape[0], seq_t.shape[0], seq_t.shape[0], pair_prev.shape[-1]), device=sampler.device)
        state_prev_ = torch.zeros((state_prev.shape[0], seq_t.shape[0], state_prev.shape[-1]), device=sampler.device)

        msa_prev_[:, roi_indices,:] = msa_prev[:, prev_roi_indices,:]
        pair_prev_[:, roi_indices, roi_indices, :] = pair_prev[:, prev_roi_indices, prev_roi_indices, :]
        #pair_prev_[:, roi_indices.reshape(-1,1), roi_indices.reshape(1,-1), :] = pair_prev[:, prev_roi_indices.reshape(-1,1), prev_roi_indices.reshape(1,-1), :]
        state_prev_[:, roi_indices,:] = state_prev[:, prev_roi_indices,:]

        msa_prev = torch.clone(msa_prev_).clone().detach()
        pair_prev = torch.clone(pair_prev_).clone().detach()
        state_prev = torch.clone(state_prev_).clone().detach()

        if sampler.inf_conf.inversion_train:
            msa_prev.requires_grad_(True)
            pair_prev.requires_grad_(True)
            state_prev.requires_grad_(True)
            optimiser = torch.optim.SGD([msa_prev, pair_prev, state_prev], lr=4e-4, momentum=0.9)

        for param in sampler.model.parameters():
            param.requires_grad = False
        print(f'Rate is: {random_rate}')
        # Loop over number of reverse diffusion time steps.
        for t in range(int(sampler.t_step_input), sampler.inf_conf.final_step - 1, -1):
            final_step = sampler.inf_conf.final_step
            msa_masked, msa_full, seq_in, xt_in, idx_pdb, t1d, t2d, xyz_t, alpha_t = sampler._preprocess(seq_t, x_t, t, inverse_ms=inverse_ms)
            if t == int(sampler.t_step_input) and i_des == design_startnum:
                pdb_coords = sampler.target_feats['xyz_27'] - torch.nanmean(sampler.target_feats['xyz_27'][original_roi_indices,:4,:], dim=(0,1)).view(1,1,3)
            if conf.inference.model_runner == 'default':
                N,L = msa_masked.shape[:2]
            else:
                B,N,L = xyz_t.shape[:3]

            xyz_t.detach_()
            msa_masked = msa_masked.detach()
            msa_full = msa_full.detach()
            seq_in = seq_in.detach()
            xt_in = xt_in.detach()
            idx_pdb = idx_pdb.detach()
            t1d = t1d.detach()
            t2d = t2d.detach()
            alpha_t = alpha_t.detach()
            ##################################
            ######## Str Self Cond ###########
            ##################################

            if conf.inference.model_runner == 'SelfConditioning':
                if (t < sampler.diffuser.T) and (t != sampler.diffuser_conf.partial_T):
                    zeros = torch.zeros(B,1,L,24,3).float().to(xyz_t.device)
                    xyz_t = torch.cat((sampler.prev_pred.unsqueeze(1),zeros), dim=-2) # [B,T,L,27,3]
                    t2d_44   = xyz_to_t2d(xyz_t) # [B,T,L,L,44]
                else:
                    xyz_t = torch.zeros_like(xyz_t)
                    t2d_44   = torch.zeros_like(t2d[...,:44])
                # No effect if t2d is only dim 44
                t2d[...,:44] = t2d_44

            t2d_44.detach_()
            xyz_t.detach_()
            t2d = t2d.detach()
            if 'zeros' in locals():
                zeros = zeros.detach()

            if sampler.symmetry is not None:
                idx_pdb, sampler.chain_idx = sampler.symmetry.res_idx_procesing(res_idx=idx_pdb)

            ####################
            ### Forward Pass ###
            ####################



            _, _, px0, _, alpha, logits, plddt = sampler.model(msa_masked,
                                msa_full,
                                seq_in,
                                xt_in,
                                idx_pdb,
                                t1d=t1d,
                                t2d=t2d,
                                xyz_t=xyz_t,
                                alpha_t=alpha_t,
                                msa_prev=msa_prev,
                                pair_prev=pair_prev,
                                state_prev=state_prev,
                                t=torch.tensor(t),
                                return_infer=True,
                                motif_mask=sampler.mask_seq.squeeze(0), #sampler.diffusion_mask.squeeze().to(sampler.device),
                                k=random_rate,
                                inverse_ms=inverse_ms, seed=i_des, row_masking=row_masking, use_checkpoint=True)

            alpha = alpha.detach()
            logits = logits.detach()
            plddt = plddt.detach()
            log.info(f'pLDDT: {plddt.mean()}')
            print(plddt[0,roi_indices])
            log.info(f'pLDDT motif: {plddt[0,roi_indices].mean()}')
            log.info(f'pLDDT motif min: {plddt[0,roi_indices].min()}')
            if conf.inference.model_runner == 'SelfConditioning':
                sampler.prev_pred = torch.clone(px0).detach()

            # prediction of X0
            _, px0  = sampler.allatom(torch.argmax(seq_in, dim=-1), px0, alpha)
            px0    = px0.squeeze()[:,:14]
            ###########################
            ### Generate Next Input ###
            ###########################

            if conf.inference.model_runner == 'SelfConditioning':
                seq_t_1 = torch.clone(seq_init)
                if t > final_step:
                    x_t_1, px0 = sampler.denoiser.get_next_pose(
                        xt=x_t,
                        px0=px0,
                        t=t,
                        pdb_coords=pdb_coords.detach(),
                        diffusion_mask=sampler.mask_str.squeeze(),
                        align_motif=sampler.inf_conf.align_motif,
                        include_motif_sidechains=sampler.preprocess_conf.motif_sidechain_input,
                        fix_motif=False,
                        original_roi_indices=original_roi_indices,
                        training=sampler.inf_conf.inversion_train,
                        inverse_ms=inverse_ms,
                    )
                    x_t_1 = x_t_1.detach()
                    sampler._log.info(
                            f'Timestep {t}, input to next step: { seq2chars(torch.argmax(seq_t_1, dim=-1).tolist())}')
                else:
                    x_t_1 = torch.clone(px0).to(x_t.device)
                    px0 = px0.to(x_t.device)

            if sampler.symmetry is not None:
                x_t_1, seq_t_1 = sampler.symmetry.apply_symmetry(x_t_1, seq_t_1)

            x_t_1 = x_t_1.detach()
            seq_t_1 = seq_t_1.detach()
            x_t = torch.clone(x_t_1)
            seq_t = torch.clone(seq_t_1).detach()

            px0_xyz_stack.append(torch.clone(px0).detach())
            denoised_xyz_stack.append(x_t)
            seq_stack.append(seq_t)
            plddt_stack.append(plddt[0].detach())  # remove singleton leading dimension

            # Optimising embeddings
            if not inverse_ms or not sampler.inf_conf.inversion_train:
                loss = loss_fn(px0[roi_indices,:4,:].to(sampler.device), pdb_coords[original_roi_indices,:4,:].detach().to(sampler.device))
            if inverse_ms and sampler.inf_conf.inversion_train:
                loss = 0.0 * loss_fn(px0[:, :4, :], pdb_coords[:, :4, :]) # pre-alignment penalisation, whole structures
                scaffold_indices = [i for i in range(pdb_coords.shape[0]) if i not in original_roi_indices]
                pdb_coords = pdb_coords.to(px0.device)

                # Extract coordinates
                pred_scaf = px0[scaffold_indices, :4, :].reshape(-1, 3)
                true_scaf = pdb_coords[scaffold_indices, :4, :].reshape(-1, 3)
                pred_roi_3d = px0[roi_indices, :4, :]  # shape: [L, 4, 3]
                true_roi_3d = pdb_coords[original_roi_indices, :4, :]  # same

                # Flattened versions for alignment only
                pred_roi_flat = pred_roi_3d.reshape(-1, 3)
                true_roi_flat = true_roi_3d.reshape(-1, 3)

                # Align predicted ROI using scaffold alignment
                pred_roi_aligned_flat = pred_roi_flat # testing 05/08/2025 (R @ pred_roi_flat.T).T + _t
                pred_roi_aligned = pred_roi_aligned_flat.reshape_as(pred_roi_3d)

                # RMSD loss (post-alignment)
                loss += 0.1 * loss_fn(pred_roi_aligned_flat, true_roi_flat) +  0.1 * loss_fn(pred_scaf, true_scaf)

                # Optionally prevent gradient updates for scaffold atoms (if training full model)
                # testing 05/08/2025 with torch.no_grad():
                # testing 05/08/2025    px0[scaffold_indices, :14, :] = pdb_coords[scaffold_indices, :14, :]
                pred_ca = pred_roi_aligned[:, 1, :]  # [N, 3]
                true_ca = true_roi_3d[:, 1, :]          # [N, 3]

                dmat_pred = torch.cdist(pred_ca, pred_ca)  # [N, N]
                dmat_true = torch.cdist(true_ca, true_ca)  # [N, N]

                # You can also mask this to include only local distances (optional)
                loss += 0.025 * loss_fn(dmat_pred, dmat_true)  # Cα–Cα shape regularisation

                def compute_backbone_torsions(coords):
                    """
                    Compute phi and psi backbone torsion angles from coordinates.
                    Args:
                        coords: [L, 4, 3] tensor (N, CA, C, O) per residue.
                    Returns:
                        phi: [L-2] tensor
                        psi: [L-2] tensor
                    """
                    N, CA, C = coords[:-1, 0], coords[:-1, 1], coords[:-1, 2]
                    N_next = coords[1:, 0]
                    CA_next = coords[1:, 1]
                    C_prev = coords[:-2, 2]
                    N_mid = coords[1:-1, 0]
                    CA_mid = coords[1:-1, 1]
                    C_mid = coords[1:-1, 2]
                    N_next2 = coords[2:, 0]

                    def torsion(p0, p1, p2, p3):
                        b0 = -1.0 * (p1 - p0)
                        b1 = p2 - p1
                        b2 = p3 - p2
                        b1 = b1 / torch.norm(b1, dim=-1, keepdim=True)
                        v = b0 - (b0 * b1).sum(-1, keepdim=True) * b1
                        w = b2 - (b2 * b1).sum(-1, keepdim=True) * b1
                        x = (v * w).sum(-1)
                        y = torch.cross(b1, v, dim=-1) * w
                        y = y.sum(-1)
                        return torch.atan2(y, x)

                    phi = torsion(C_prev, N_mid, CA_mid, C_mid)
                    psi = torsion(N_mid, CA_mid, C_mid, N_next2)
                    return phi, psi

                def torsion_angle_loss(pred_coords, true_coords):
                    """
                    Compute torsion loss based on phi and psi angles.
                    Args:
                        pred_coords, true_coords: [L, 4, 3] tensors
                    Returns:
                        scalar loss
                    """
                    phi_pred, psi_pred = compute_backbone_torsions(pred_coords)
                    phi_true, psi_true = compute_backbone_torsions(true_coords)

                    loss_phi = torch.mean(1.0 - torch.cos(phi_pred - phi_true))
                    loss_psi = torch.mean(1.0 - torch.cos(psi_pred - psi_true))
                    return loss_phi + loss_psi
                torsion_loss = torsion_angle_loss(px0[roi_indices, :4, :], pdb_coords[original_roi_indices, :4, :])
                loss += 0.2 * torsion_loss

            loss_list.append(loss.item())
            if sampler.inf_conf.inversion_train:
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            if t == 1 and sampler.inf_conf.inversion_train:
                torch.save(state_prev[:, roi_indices, :], os.path.join(embeddings_dir, f'state_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
                torch.save(pair_prev[:, roi_indices.reshape(-1,1), roi_indices, :], os.path.join(embeddings_dir, f'pair_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
                torch.save(msa_prev[:, roi_indices, :], os.path.join(embeddings_dir, f'msa_{sampler.inf_conf.input_pdb[11:-4]}.pt'))
                print(loss_list)

            torch.cuda.empty_cache()
            px0.detach_()

            print(f'Timestep {t}, loss: {loss.item()}')

        # Flip order for better visualization in pymol
        denoised_xyz_stack = torch.stack(denoised_xyz_stack)
        denoised_xyz_stack = torch.flip(
            denoised_xyz_stack,
            [
                0,
            ],
        )
        px0_xyz_stack = torch.stack(px0_xyz_stack)
        px0_xyz_stack = torch.flip(
            px0_xyz_stack,
            [
                0,
            ],
        )

        # For logging -- don't flip
        plddt_stack = torch.stack(plddt_stack)

        # Save outputs
        os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
        final_seq = seq_stack[-1]

        # Output glycines, except for motif region
        final_seq = torch.where(
            torch.argmax(seq_init, dim=-1) == 21, 7, torch.argmax(seq_init, dim=-1)
        )  # 7 is glycine

        bfacts = torch.ones_like(final_seq.squeeze())
        # make bfact=0 for diffused coordinates
        bfacts[torch.where(torch.argmax(seq_init, dim=-1) == 21, True, False)] = 0
        # pX0 last step
        out = f"{out_prefix}.pdb"

        # Now don't output sidechains
        if not inverse_ms:
            writepdb(
                out,
                denoised_xyz_stack[0, :, :4],
                final_seq,
                sampler.binderlen,
                chain_idx=sampler.chain_idx,
                bfacts=bfacts,
            )
        else:
            writepdb(
                out,
                denoised_xyz_stack[0, :, :14],
                final_seq,
                sampler.binderlen,
                chain_idx=sampler.chain_idx,
                bfacts=bfacts,
            )

        # run metadata
        trb = dict(
            config=OmegaConf.to_container(sampler._conf, resolve=True),
            plddt=plddt_stack.cpu().numpy(),
            device=torch.cuda.get_device_name(torch.cuda.current_device())
            if torch.cuda.is_available()
            else "CPU",
            time=time.time() - start_time,
        )
        if hasattr(sampler, "contig_map"):
            for key, value in sampler.contig_map.get_mappings().items():
                trb[key] = value
        log.info(f"Finished design in {(time.time()-start_time)/60:.2f} minutes")



In [ ]:
run_inversion(config_path="../config/inference", config_name="base",
              overrides=["inference.num_designs=5"],
              output_dir="./outputs/my_run")

NameError: name '__file__' is not defined

: 

In [23]:

import os

from hydra import compose, initialize_config_dir

repo = Path("~/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL").expanduser().resolve()

config_dir = '/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/config/inference/'

input_pdb = '/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/examples/input_pdbs/5UDC_H_only.pdb'

In [27]:
#contig='0/H94-114/0 H1-93/H115-217/0 L1-211/0 F1-449/0'
contig='92-92/H93-114/103-103'
# Hydra's list override needs to be a single string.
overrides = [
    f"inference.input_pdb={input_pdb}",
    "inference.output_prefix=outputs/nirsevimab/pgel_train",
    "inference.num_designs=3",
    "inference.inversion_train=True",
    "inference.cautious=False",
    f"contigmap.contigs=[{contig}]",
]

with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    conf = compose(config_name="base", overrides=overrides)

print(conf.inference.input_pdb)
print(conf.contigmap.contigs)

/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/examples/input_pdbs/5UDC_H_only.pdb
['92-92/H93-114/103-103']


In [31]:
from pathlib import Path
from omegaconf import OmegaConf

new_pdb = Path(
    input_pdb
).resolve()

new_contig = contig  # your new contig

OmegaConf.set_struct(conf, False)

OmegaConf.update(
    conf,
    "inference.input_pdb",
    str(new_pdb),
    force_add=True,
)

conf.contigmap.contigs = [new_contig]

OmegaConf.update(
    conf,
    "inference.trb_save_ckpt_path",
    None,
    force_add=True,
)

print("Configured PDB:", conf.inference.input_pdb)
print("Configured contig:", conf.contigmap.contigs)

Configured PDB: /home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/examples/input_pdbs/5UDC_H_only.pdb
Configured contig: ['92-92/H93-114/103-103']


In [32]:
from omegaconf import OmegaConf

OmegaConf.set_struct(conf, False)

# This field must be null; it is only populated internally by PGEL.
OmegaConf.update(
    conf,
    "inference.trb_save_ckpt_path",
    None,
    force_add=True,
)

print("trb_save_ckpt_path =", conf.inference.trb_save_ckpt_path)

run_new_inversion(conf)

trb_save_ckpt_path = None
Reading models from /home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/rfdiffusion/inference/../../models
This is inf_conf.ckpt_path
/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/rfdiffusion/inference/../../models/Base_ckpt.pt


/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/PGEL/rfdiffusion/inference/model_runners.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.ckpt  = torch.load(


Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_msa] = 256
USING MODEL CONFIG: self._conf[model][d_msa_full] = 64
USING MODEL CONFIG: self._conf[model][d_pair] = 128
USING MODEL CONFIG: self._conf[model][d_templ] = 64
USING MODEL CONFIG: self._conf[model][n_head_msa] = 8
USING MODEL CONFIG: self._conf[model][n_head_pair] = 4
USING MODEL CONFIG: self._conf[model][n_head_templ] = 4
USING MODEL CONFIG: self._conf[model][d_hidden] = 32
USING MODEL CONFIG: self._conf[model][d_hidden_templ] = 32
USING MODEL CONFIG: self._conf[model][p_drop] = 0.15
USING MODEL CONFIG: self._conf[model][SE3_param_full] = {'num_layers': 1, 'num_channels': 32, 'num_degrees': 2, 'n_heads': 4, 'div': 4, 'l0_in_features': 8, 'l0_out_features': 8, 'l1_in_features': 3, 'l1_out_features': 2, 

OutOfMemoryError: CUDA out of memory. Tried to allocate 70.00 MiB. GPU 0 has a total capacity of 11.59 GiB of which 56.25 MiB is free. Process 41193 has 143.47 MiB memory in use. Process 41781 has 198.22 MiB memory in use. Including non-PyTorch memory, this process has 10.60 GiB memory in use. Of the allocated memory 10.22 GiB is allocated by PyTorch, and 46.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

def parse_tm_scores(output: str) -> list[float]:
    scores = [float(x) for x in TM_SCORE_RE.findall(output)]
    if not scores:
        raise RuntimeError("Could not find TM-score in TM-align/USalign output")
    return scores


def run_tm_align(executable: str, a: Path, b: Path, timeout: int, mode: str) -> float:
    result = subprocess.run(
        [executable, str(a), str(b)],
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )
    output = result.stdout + "\n" + result.stderr
    if result.returncode != 0:
        raise RuntimeError(f"TM-align failed for {a.name} vs {b.name}:\n{output}")
    scores = parse_tm_scores(output)
    if mode == "max":
        return max(scores)
    if mode == "mean":
        return float(np.mean(scores))
    return scores[0]


def pairwise_tm_scores(paths: list[Path], executable: str, timeout: int, mode: str) -> np.ndarray:
    n = len(paths)
    matrix = np.eye(n, dtype=float)
    total = n * (n - 1) // 2
    done = 0
    for i in range(n):
        for j in range(i + 1, n):
            score = run_tm_align(executable, paths[i], paths[j], timeout, mode)
            matrix[i, j] = matrix[j, i] = score
            done += 1
            print(f"[{done}/{total}] {paths[i].name} vs {paths[j].name}: {score:.4f}", file=sys.stderr)
    return matrix


def extract_region(
    source: Path,
    destination: Path,
    chain: str,
    start: int,
    end: int,
    output_chain: str | None = None,
) -> None:
    """Extract protein records for chain/start:end from a PDB.

    If output_chain is supplied, rewrite the extracted records to that chain
    ID. This is useful when PGEL outputs use a different chain ID from the
    native input, e.g. candidate chain A versus native chain H.
    """
    kept = []
    output_chain = output_chain or chain
    if len(output_chain) != 1:
        raise ValueError("Chain IDs must be one character")

    with source.open() as handle:
        for line in handle:
            if not (line.startswith("ATOM  ") or line.startswith("HETATM")):
                continue
            if len(line) < 26 or line[21].strip() != chain:
                continue
            if line[17:20].strip() not in AA3:
                continue
            try:
                residue_number = int(line[22:26])
            except ValueError:
                continue
            if start <= residue_number <= end:
                rewritten = line[:21] + output_chain + line[22:66].rstrip() + "\n"
                kept.append(rewritten)

    if not kept:
        raise ValueError(f"No residues found in {source}: chain {chain}, {start}-{end}")
    destination.write_text("".join(kept) + "END\n")


def motif_tm_scores(
    candidates: list[Path],
    native_pdb: Path,
    executable: str,
    timeout: int,
    mode: str,
    native_chain: str,
    candidate_chain: str,
    start: int,
    end: int,
) -> np.ndarray:
    with tempfile.TemporaryDirectory(prefix="tm_motif_") as tmp:
        tmpdir = Path(tmp)
        native_motif = tmpdir / "native_motif.pdb"
        extract_region(native_pdb, native_motif, native_chain, start, end, output_chain="H")
        values = []
        for candidate in candidates:
            candidate_motif = tmpdir / f"{candidate.stem}_motif.pdb"
            extract_region(candidate, candidate_motif, candidate_chain, start, end, output_chain="H")
            values.append(run_tm_align(executable, candidate_motif, native_motif, timeout, mode))
            print(f"{candidate.name}: native motif TM-score={values[-1]:.4f}", file=sys.stderr)
        return np.asarray(values, dtype=float)


def cluster_at_threshold(tm_matrix: np.ndarray, tm_threshold: float, method: str) -> tuple[np.ndarray, object | None]:
    if not 0.0 < tm_threshold <= 1.0:
        raise ValueError("TM thresholds must be > 0 and <= 1")
    distance = 1.0 - tm_matrix
    n = len(tm_matrix)
    if n == 1:
        return np.ones(1, dtype=int), None
    tree = linkage(squareform(distance, checks=False), method=method)
    clusters = fcluster(tree, t=1.0 - tm_threshold, criterion="distance")
    return clusters, tree


def save_dendrogram(tree, labels: list[str], tm_threshold: float, output: Path) -> None:
    plt.figure(figsize=(max(10, 0.35 * len(labels)), 7))
    if tree is None:
        plt.text(0.5, 0.5, labels[0], ha="center", va="center", fontsize=12)
        plt.xlim(0, 1)
        plt.ylim(0, 1)
    else:
        cutoff = 1.0 - tm_threshold
        dendrogram(tree, labels=labels, color_threshold=cutoff, leaf_rotation=90, leaf_font_size=8)
        plt.axhline(cutoff, color="black", linestyle="--", linewidth=1,
                    label=f"TM-score threshold = {tm_threshold:.3f}")
        plt.legend()
    plt.ylabel("Dissimilarity (1 - TM-score)")
    plt.title("Hierarchical clustering of predicted structures")
    plt.tight_layout()
    plt.savefig(output, dpi=300)
    plt.close()


def write_matrix(path: Path, labels: list[str], matrix: np.ndarray, header: str) -> None:
    with path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow([header, *labels])
        for label, row in zip(labels, matrix):
            writer.writerow([label, *[f"{x:.8f}" for x in row]])



In [ ]:

import csv
import sys
import numpy as np
from __future__ import annotations

import argparse
import csv
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform

TM_SCORE_RE = re.compile(r"TM-score\s*=\s*([0-9]*\.?[0-9]+)")
AA3 = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS",
    "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP",
    "TYR", "VAL",
}


def find_pdbs(input_dir: Path, pattern: str) -> list[Path]:
    paths = sorted(input_dir.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No structures matched {input_dir / pattern}")
    return paths


def run_tm_diversity(
    input_dir=None,
    pattern="*.pdb",
    tmalign="TM-align",
    tm_thresholds=(0.70,),
    linkage="average",
    score_mode="max",
    timeout=300,
    output_prefix=None,
    native_pdb=None,
    motif_chain="H",
    candidate_motif_chain=None,
    motif_range=None,
):
    """Run TM-score diversity analysis directly from a notebook.

    The helper functions find_pdbs, pairwise_tm_scores, motif_tm_scores,
    cluster_at_threshold, save_dendrogram, and write_matrix must already be
    defined or imported in the notebook.
    """
    repo = Path("~/Documents/FirstYear/PGEL_pipeline/PGEL").expanduser().resolve()

    # Default to the PGEL outputs produced by the previous sampling step.
    if input_dir is None:
        input_dir = repo / "examples" / "outputs" / "nirsevimab"
    else:
        input_dir = Path(input_dir).expanduser().resolve()

    if output_prefix is None:
        output_prefix = input_dir / "tm_diversity"
    else:
        output_prefix = Path(output_prefix).expanduser().resolve()

    output_prefix.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(tm_thresholds, str):
        thresholds = [float(x.strip()) for x in tm_thresholds.split(",")]
    else:
        thresholds = [float(x) for x in tm_thresholds]

    paths = find_pdbs(input_dir, pattern)
    labels = [p.stem for p in paths]
    print(f"Found {len(paths)} structures", file=sys.stderr)

    if len(paths) == 0:
        raise FileNotFoundError(
            f"No PDB files matching {pattern!r} found in {input_dir}"
        )

    tm_matrix = pairwise_tm_scores(paths, tmalign, timeout, score_mode)
    write_matrix(
        Path(f"{output_prefix}_pairwise_tm.csv"),
        labels,
        tm_matrix,
        "structure",
    )

    native_scores = None
    if native_pdb is not None or motif_range is not None:
        if native_pdb is None or motif_range is None:
            raise ValueError(
                "native_pdb and motif_range must be supplied together"
            )

        native_pdb = Path(native_pdb).expanduser().resolve()
        try:
            start, end = map(int, str(motif_range).split("-"))
        except ValueError as exc:
            raise ValueError(
                "motif_range must look like 'START-END', for example '83-122'"
            ) from exc

        candidate_chain = candidate_motif_chain or motif_chain
        print(
            f"Extracting native motif from chain {motif_chain}; "
            f"candidate motifs from chain {candidate_chain}; "
            f"rewriting both temporary motifs to chain H",
            file=sys.stderr,
        )

        native_scores = motif_tm_scores(
            paths,
            native_pdb,
            tmalign,
            timeout,
            score_mode,
            motif_chain,
            candidate_chain,
            start,
            end,
        )

        with Path(f"{output_prefix}_native_motif_tm.csv").open(
            "w", newline=""
        ) as handle:
            writer = csv.writer(handle)
            writer.writerow(["structure", "native_motif_tm_score"])
            writer.writerows(
                (p.name, f"{x:.8f}")
                for p, x in zip(paths, native_scores)
            )

    summary_rows = []
    for threshold in thresholds:
        clusters, tree = cluster_at_threshold(
            tm_matrix,
            threshold,
            linkage,
        )
        cluster_count = len(set(clusters))
        dendro = Path(
            f"{output_prefix}_tm{threshold:.3f}_dendrogram.png"
        )
        save_dendrogram(tree, labels, threshold, dendro)

        assignments = Path(
            f"{output_prefix}_tm{threshold:.3f}_clusters.csv"
        )
        with assignments.open("w", newline="") as handle:
            writer = csv.writer(handle)
            header = ["structure", "cluster"]
            if native_scores is not None:
                header += [
                    "native_motif_tm_score",
                    "cluster_distinguishable_from_native",
                ]
            writer.writerow(header)

            for i, (p, cluster) in enumerate(zip(paths, clusters)):
                row = [p.name, int(cluster)]
                if native_scores is not None:
                    members = np.where(clusters == cluster)[0]
                    distinguishable = bool(
                        np.min(native_scores[members]) < threshold
                    )
                    row += [
                        f"{native_scores[i]:.8f}",
                        distinguishable,
                    ]
                writer.writerow(row)

        if native_scores is None:
            distinguishable_clusters = ""
        else:
            distinguishable_clusters = sum(
                np.min(native_scores[clusters == c]) < threshold
                for c in set(clusters)
            )

        summary_rows.append([
            threshold,
            cluster_count,
            distinguishable_clusters,
        ])
        print(
            f"TM threshold {threshold:.3f}: {cluster_count} clusters",
            file=sys.stderr,
        )

    with Path(f"{output_prefix}_summary.csv").open(
        "w", newline=""
    ) as handle:
        writer = csv.writer(handle)
        writer.writerow([
            "tm_threshold",
            "n_clusters",
            "n_clusters_distinguishable_from_native",
        ])
        writer.writerows(summary_rows)

    print(f"Wrote {output_prefix}_pairwise_tm.csv")
    if native_scores is not None:
        print(f"Wrote {output_prefix}_native_motif_tm.csv")
    print(f"Wrote {output_prefix}_summary.csv")

    return {
        "paths": paths,
        "tm_matrix": tm_matrix,
        "native_scores": native_scores,
        "output_prefix": output_prefix,
    }


In [ ]:
import shutil
import subprocess

for name in ["TM-align", "TMalign", "USalign"]:
    print(name, "->", shutil.which(name))

TM-align -> None
TMalign -> /home/ics121/local/bin/tmalign/TMalign
USalign -> None


In [ ]:

tmalign_executable = shutil.which("TMalign") or shutil.which("TM-align") or shutil.which("USalign")

if tmalign_executable is None:
    raise FileNotFoundError(
        "No TM-align/TMalign/USalign executable found on PATH."
    )


In [15]:
results = run_tm_diversity(
    input_dir=(
        "/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/"
        "PGEL/notebooks/5UDC_H_only_outputs/nirsevimab"
    ),
    tmalign="TMalign",
    tm_thresholds=[0.70],
    native_pdb=(
        "/home/ics121/Documents/FirstYear/PGEL_pipeline/PGEL/"
        "PGEL/examples/input_pdbs/5UDC_H_only.pdb"
    ),
    motif_chain="H",              # native PDB chain
    candidate_motif_chain="A",    # PGEL output chain
    motif_range="83-122",
)

NameError: name 'run_tm_diversity' is not defined